## Setup

In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

if not os.getcwd().endswith("etl"):
    os.chdir("./etl")

import requests
import pandas as pd
import pyspark
from io import BytesIO
from dotenv import load_dotenv
from json import load
from pyspark.sql import functions as F, Window
from tqdm import tqdm
from time import sleep

In [4]:
load_dotenv()

JDBC_PATH = os.getenv("JDBC_PATH")
_DB_URL = os.getenv("DB_URL")
_DB_NAME = os.getenv("DB_NAME")
_DB_USER = os.getenv("DB_USER")
_DB_PASSWORD = os.getenv("DB_PASSWORD")

if JDBC_PATH is None or _DB_URL is None or _DB_NAME is None or _DB_USER is None or _DB_PASSWORD is None:
    raise EnvironmentError("Set .env vars!")
if not os.path.exists(JDBC_PATH) or not os.path.isfile(JDBC_PATH):
    raise FileNotFoundError("JDBC driver not found!")

DB_URL = f"jdbc:mysql://{_DB_URL}/{_DB_NAME}"
DB_PROPERTIES = {"user": _DB_USER, "password": _DB_PASSWORD, "driver": "com.mysql.cj.jdbc.Driver"}

In [5]:
spark = pyspark.sql.SparkSession.builder.appName("ETL TCC").config("spark.jars", JDBC_PATH).getOrCreate()

26/04/21 21:36:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 21:36:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Extract

### Base bueiros

In [7]:
BUEIROS_URL = "https://www.der.sp.gov.br/WebSite/Arquivos/DadosAbertos/AtivosRodoviarios/Bueiros/Bueiros.xlsx"
res = requests.get(BUEIROS_URL)
res.raise_for_status()

In [8]:
content_wrapper = BytesIO(res.content)
pandas_df = pd.read_excel(content_wrapper)
df_raw_bueiros = spark.createDataFrame(pandas_df)

### Mocks

In [9]:
pandas_df = pd.read_csv("./mocks/distritos.csv").reset_index()
df_mock_distritos = spark.createDataFrame(pandas_df)

## Transform

#### 1. Distrito

In [81]:
df_transf_distritos = df_mock_distritos.withColumnRenamed("index", "id").withColumn("cidade", F.lit("São Paulo"))

#### 2. Coordenada

In [121]:
df_raw_coords = df_transf_distritos.toPandas()
df_raw_coords["latitude"] = 0.0 
df_raw_coords["longitude"] = 0.0 

geolocator = Nominatim(user_agent="tcc")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

for i in tqdm(df_raw_coords.index):
    bairro = df_raw_coords.loc[i, "bairro"]

    location = geocode(f"{bairro}, São Paulo")
    df_raw_coords.loc[i, "latitude"] = location.latitude
    df_raw_coords.loc[i, "longitude"] = location.longitude

df_raw_coords = df_raw_coords.reset_index().drop(columns=["cidade"]).rename(columns={"id": "endereco_id"}).rename(columns={"index": "id"})
df_transf_coords = spark.createDataFrame(df_raw_coords)
df_transf_coords.show()

100%|██████████| 96/96 [01:35<00:00,  1.01it/s]


+---+-----------+-----------------+-----------+-----------+
| id|endereco_id|           bairro|   latitude|  longitude|
+---+-----------+-----------------+-----------+-----------+
|  0|          0|           Grajaú|-23.7799713|-46.6737655|
|  1|          1|    Jardim Ângela|-23.7125278|-46.7687195|
|  2|          2|    Capão Redondo|-23.6719026|-46.7794354|
|  3|          3|        Sapopemba|-23.6043265|-46.5098851|
|  4|          4|           Sacomã|-23.6012824|-46.6025552|
|  5|          5|  Jardim São Luís|-23.6739274|-46.7407377|
|  6|          6|    Cidade Ademar|-23.6730116|-46.6552806|
|  7|          7|      Brasilândia|-23.4649661|-46.6913891|
|  8|          8|      Campo Limpo|-23.6361999|-46.7627968|
|  9|          9|        Jabaquara|-23.6463382|-46.6410462|
| 10|         10|          Jaraguá| -23.446592|-46.7361775|
| 11|         11|         Itaquera|-23.5422994|-46.4712065|
| 12|         12|   Itaim Paulista|-23.5017648|-46.3996091|
| 13|         13|         Tremembé|-22.9

#### 3. Tipo

In [122]:
df_transf_tipo = df_raw_bueiros.select("Tipo").drop_duplicates().withColumnRenamed("Tipo", "nome")\
    .withColumn("id", F.row_number().over(Window.orderBy(F.lit(1))))

#### 4. Bueiro

In [123]:
# df_transf_bueiro = df_raw_bueiros.withColumnRenamed("Extensão (m)", "extensao").withColumnRenamed("Dimensão (m)", "dimensao")\
#     .withColumnRenamed("Tipo", "nome")\
#     .withColumn("endereco", F.concat(F.col("Rodovia"), F.lit(" KM"), F.col("KM").cast("string"))).select("extensao", "dimensao", "nome", "endereco")

In [124]:
# df_pandas_bueiro = df_transf_bueiro.toPandas()
# df_pandas_bueiro["bairro"] = ""

# geolocator = Nominatim(user_agent="tcc")
# reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

# for i in tqdm(df_pandas_bueiro.index):
#     endereco = df_pandas_bueiro.loc[i, "endereco"]

#     location = geocode(endereco)
#     if location is None:
#         continue
#     if "address" not in location.raw.keys():
#         continue
#     print(location)
#     df_pandas_bueiro.loc[i, "bairro"] = bairro
#     break

# df_pandas_bueiro

In [125]:
# df_transf_bueiro = df_transf_bueiro.join(df_transf_tipo, on="nome").withColumnRenamed("id", "tipo_id").drop("nome")
# df_transf_bueiro = df_transf_bueiro.join(df_transf_distritos, on="bairro").withColumnRenamed("id", "endereco_id").drop("bairro", "cidade")

# df_transf_bueiro = df_transf_bueiro.withColumn("id", F.row_number().over(Window.orderBy(F.lit(1))))

#### 5. Chuva e solo

In [ ]:
OPEN_METEO_BASE_URL = "https://api.open-meteo.com/v1/forecast?latitude={}&longitude={}&hourly=rain&past_days=31&forecast_days=1"
lats_longs_distr = df_transf_coords.drop("id").withColumnRenamed("endereco_id", "id").join(df_transf_distritos, on="id").select("id", "latitude", "longitude")\
    .withColumnRenamed("id", "distrito_id").drop_duplicates().toPandas()

full_hourly_rain = []
chunk_size = 50
for i in tqdm(lats_longs_distr.index):
    distrito_id = lats_longs_distr.loc[i, "distrito_id"]
    lat = lats_longs_distr.loc[i, "latitude"]
    long = lats_longs_distr.loc[i, "longitude"]
    
    res = requests.get(OPEN_METEO_BASE_URL.format(lat, long))
    res.raise_for_status()
    
    subset_hourly_rain = load(BytesIO(res.content))
    subset_hourly_rain["distrito_id"] = distrito_id
    full_hourly_rain.append(subset_hourly_rain)

  5%|▌         | 5/96 [00:02<00:39,  2.28it/s]                                  

In [ ]:
df_raw_rain = pd.DataFrame.from_records(full_hourly_rain)[["distrito_id", "latitude", "longitude", "hourly"]]
df_raw_rain["time"] = df_raw_rain["hourly"].apply(lambda x: x["time"])
df_raw_rain["rain"] = df_raw_rain["hourly"].apply(lambda x: x["rain"])
df_raw_rain = df_raw_rain.drop(columns="hourly").explode(["time", "rain"])

df_raw_rain = df_raw_rain[df_raw_rain["rain"] > 0]

df_raw_rain["group_id"] = df_raw_rain['rain'].ne(df_raw_rain['rain'].shift()).cumsum()

df_agg_rain = df_raw_rain.groupby(["distrito_id", "latitude", "longitude", "group_id"]).agg(
    dt_hr_inicio=('time', 'first'), dt_hr_fim=('time', 'last'), qtd=('rain', 'first')
).reset_index()

In [ ]:
df_transf_rain = spark.createDataFrame(df_agg_rain).drop("group_id", "latitude", "longitude").withColumn("id", F.row_number().over(Window.orderBy(F.lit(1))))

#### 6. Complemento coordenada

In [ ]:
df_pandas_coords = df_transf_coords.toPandas().drop(columns="endereco_id").rename(columns={"id":"coordenada_id"})
df_pandas_coords["rua"] = ""
df_pandas_coords["numero"] = ""
df_pandas_coords["cep"] = ""

geolocator = Nominatim(user_agent="tcc")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

for i in tqdm(df_pandas_coords.index):
    lat = df_pandas_coords.loc[i, "latitude"]
    long = df_pandas_coords.loc[i, "longitude"]

    location = reverse((lat, long))
    if location is not None:
        address = location.raw["address"]

    df_pandas_coords.loc[i, "rua"] = address["road"]
    if "house_number" in address.keys():
        df_pandas_coords.loc[i, "numero"] = address["house_number"]
    if "postcode" in address.keys():
        df_pandas_coords.loc[i, "cep"] = address["postcode"].replace("-", "")

100%|██████████| 96/96 [01:35<00:00,  1.01it/s]


In [ ]:
df_tranf_compl_coord = spark.createDataFrame(df_pandas_coords.reset_index().drop(columns=["latitude", "longitude"]).rename(columns={"index": "id"}))

## Load

### 1. Distrito

In [ ]:
df_transf_distritos.write.jdbc(DB_URL, "distrito", properties=DB_PROPERTIES, mode="append")

### 2. Coordenada

In [ ]:
df_transf_coords.write.jdbc(DB_URL, "coordenada", properties=DB_PROPERTIES, mode="append")

26/04/21 22:12:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


### 3. Tipo

In [ ]:
df_transf_tipo.write.jdbc(DB_URL, "tipo", properties=DB_PROPERTIES, mode="append")

26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 2

### 4. Bueiro

In [ ]:
# df_transf_bueiro.write.jdbc(DB_URL, "bueiro", properties=DB_PROPERTIES, mode="append")

26/04/21 22:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 2

### 5. Chuva

In [ ]:
df_transf_rain.write.jdbc(DB_URL, "chuva", properties=DB_PROPERTIES, mode="append")

26/04/21 22:12:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 22:12:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


### 6. Complemento coordenadas

In [ ]:
df_tranf_compl_coord.write.jdbc(DB_URL, "complemento_coordenada", properties=DB_PROPERTIES, mode="append")